In [1]:
import dataclasses

import openpi.models.pi0_config
import openpi.models_pytorch.pi0_pytorch
from openpi.policies import libero_policy
from openpi.training import config as _config
# from openpi.policies import policy_config as _policy_config
# from openpi.shared import download
# from openpi.training import config as _config
# from openpi.training import data_loader as _data_loader

In [2]:
device = "cuda"
config = _config.get_config("pi05_libero")

In [3]:
model_cfg = openpi.models.pi0_config.Pi0Config(
    dtype=config.pytorch_training_precision,
    action_dim=config.model.action_dim,
    action_horizon=config.model.action_horizon,
    max_token_len=config.model.max_token_len,
    paligemma_variant=getattr(config.model, "paligemma_variant", "gemma_2b"),
    action_expert_variant=getattr(config.model, "action_expert_variant", "gemma_300m"),
    pi05=getattr(config.model, "pi05", False),
    debug=True
)

In [4]:
model = openpi.models_pytorch.pi0_pytorch.DistilledPI0Pytorch(model_cfg).to(device)
teacher_model = openpi.models_pytorch.pi0_pytorch.DistilledPI0Pytorch(model_cfg).to(device)

In [ ]:
observation, actions = libero_policy.make_libero_example_forward(model, batch_size=2, right_wrist_present=False)
loss = model(observation, actions, teacher=teacher_model)


[DBG] DistilledPI0Pytorch.forward call_id=1 train=True pi05=True
[DBG] actions: shape=(2, 10, 32) dtype=float32 device=cuda:0 req_grad=False
[DBG] state: shape=(2, 32) dtype=float32 device=cuda:0 req_grad=False
[DBG] images(list): list len=3
  [DBG] images(list)[0]: shape=(2, 3, 224, 224) dtype=float32 device=cuda:0 req_grad=False
  [DBG] images(list)[1]: shape=(2, 3, 224, 224) dtype=float32 device=cuda:0 req_grad=False
  [DBG] images(list)[2]: shape=(2, 3, 224, 224) dtype=float32 device=cuda:0 req_grad=False
[DBG] img_masks(list): list len=3
  [DBG] img_masks(list)[0]: shape=(2,) dtype=bool device=cuda:0 req_grad=False
  [DBG] img_masks(list)[1]: shape=(2,) dtype=bool device=cuda:0 req_grad=False
  [DBG] img_masks(list)[2]: shape=(2,) dtype=bool device=cuda:0 req_grad=False
[DBG] lang_tokens: shape=(2, 16) dtype=int64 device=cuda:0 req_grad=False
[DBG] lang_masks: shape=(2, 16) dtype=bool device=cuda:0 req_grad=False
[DBG] noise(sampled): shape=(2, 10, 32) dtype=float32 device=cuda:0

# Policy inference

The following example shows how to create a policy from a checkpoint and run inference on a dummy example.

In [ ]:
config = _config.get_config("pi05_libero")
checkpoint_dir = "./checkpoints/foundation/pi05_libero/pytorch"

# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = libero_policy.make_libero_example()
result = policy.infer(example)

# Delete the policy to free up memory.
del policy

print("Actions shape:", result["actions"].shape)

# Working with a live model


The following example shows how to create a live model from a checkpoint and compute training loss. First, we are going to demonstrate how to do it with fake data.


In [ ]:
config = _config.get_config("pi0_aloha_sim")

checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_aloha_sim")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)
print("Loss shape:", loss.shape)

Now, we are going to create a data loader and use a real batch of training data to compute the loss.

In [ ]:
# Reduce the batch size to reduce memory usage.
config = dataclasses.replace(config, batch_size=2)

# Load a single batch of data. This is the same data that will be used during training.
# NOTE: In order to make this example self-contained, we are skipping the normalization step
# since it requires the normalization statistics to be generated using `compute_norm_stats`.
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory.
del model

print("Loss shape:", loss.shape)